In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_file_path = '/content/drive/MyDrive/DS-studia/projekt-grupowy/CW03carrier.xlsx'

df = pd.read_excel(
    data_file_path,
    dtype_backend='numpy_nullable',
    skiprows=[0,2],
    na_values=['-'],
)

In [25]:
n = df.shape[0]
p = df.shape[1]
print(f'samples  n={n}\nfeatures p={p}')

samples  n=13506
features p=65


In [ ]:
df.columns

Index(['Timestamp', 'Pallet to Pallet CycleTime (Sec)', 'Serial Number',
       'Part Number', 'Image', 'Status', 'Went Thru Tear Down?', 'Cycles',
       'Reject Desc', 'Reject Code', 'Nest ID Verify Cycles', 'Nest ID Verify',
       'MARPOSS Judge Result', 'M9 Stretch Dist RG Side',
       'M81 DISTANCE 81,0 G', 'M81 DISTANCE 81,0 F', 'M8 CC10 DB Circularity',
       'M7 F Diam To D-E', 'M6 DB Diam To A_D - E_B', 'M58', 'M57', 'M56 P12',
       'M55 P11', 'M54 P6', 'M53 P5', 'M52', 'M51', 'M50',
       'M5 CC9 Diff Bore Diam', 'M45 OFFSET 38,1 G SIDE',
       'M44 OFFSET 38,1 F SIDE', 'M4 SB Diam To F-G/A 2',
       'M37 CC16 F-G To D-E TP', 'M36 OD Flange To D-E TP', 'M35 OD Flange',
       'M34 TD Flange To D-E TP', 'M33 TD Flange Diam',
       'M32 CC2 PN Seal To D-E', 'M31 PN Seal Diam', 'M30 E To A PN TP',
       'M3 SB Diam To F-G/A 1', 'M29 E-D To A-B-C PN TP',
       'M28 PN TB Circularity', 'M27 PN TB Diam', 'M26 PN HB Circularity',
       'M25 CC11 PN HB Diam', 'M24 F-G To 

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13506 entries, 0 to 13505
Data columns (total 65 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Timestamp                         13506 non-null  datetime64[ns]
 1   Pallet to Pallet CycleTime (Sec)  13505 non-null  Int64         
 2   Serial Number                     13506 non-null  string        
 3   Part Number                       13506 non-null  string        
 4   Image                             0 non-null      Int64         
 5   Status                            13506 non-null  string        
 6   Went Thru Tear Down?              515 non-null    string        
 7   Cycles                            13506 non-null  Int64         
 8   Reject Desc                       199 non-null    string        
 9   Reject Code                       13506 non-null  Int64         
 10  Nest ID Verify Cycles             13505 non-nu

`Serial Number` i `Part Number` powinny być numerami - gdzieś uszkodzone wpisy.

`Image` brak informacji. Do wyrzucenia.

`Status` zmienna kategorialna - bardzo interesująca, być może zmienna `target`.

`Reject Desc` opis, może się przydać, ale raczej nie jest znormalizowana. Możliwe że powtarza binarną informację ze `Status`

`Reject Code` opis odrzutu ale znormalizowany. Bardzo ciekawe. Może korelować z innymi zmiennymi które wskazują na przekroczenia tolerancji lub są predyktorami odrzutu.

In [46]:
print(f'Więcej niż 1000 braków: {df.columns[(df.isnull().sum() > 1000).values]}')
print(f'Więcej niż 1 braków: {df.columns[(df.isnull().sum() > 1).values]}')
print(f'Więcej niż 0 braków: {df.columns[(df.isnull().sum() > 0).values]}')

Więcej niż 1000 braków: Index(['Image', 'Went Thru Tear Down?', 'Reject Desc'], dtype='object')
Więcej niż 1 braków: Index(['Image', 'Went Thru Tear Down?', 'Reject Desc'], dtype='object')
Więcej niż 0 braków: Index(['Pallet to Pallet CycleTime (Sec)', 'Image', 'Went Thru Tear Down?',
       'Reject Desc', 'Nest ID Verify Cycles', 'MARPOSS Judge Result',
       'M9 Stretch Dist RG Side', 'M81 DISTANCE 81,0 G', 'M81 DISTANCE 81,0 F',
       'M8 CC10 DB Circularity', 'M7 F Diam To D-E', 'M6 DB Diam To A_D - E_B',
       'M58', 'M57', 'M56 P12', 'M55 P11', 'M54 P6', 'M53 P5', 'M52', 'M51',
       'M50', 'M5 CC9 Diff Bore Diam', 'M45 OFFSET 38,1 G SIDE',
       'M44 OFFSET 38,1 F SIDE', 'M4 SB Diam To F-G/A 2',
       'M36 OD Flange To D-E TP', 'M35 OD Flange', 'M34 TD Flange To D-E TP',
       'M33 TD Flange Diam', 'M32 CC2 PN Seal To D-E', 'M31 PN Seal Diam',
       'M30 E To A PN TP', 'M3 SB Diam To F-G/A 1', 'M29 E-D To A-B-C PN TP',
       'M28 PN TB Circularity', 'M27 PN TB Diam', 'M

Zapewne jest jakiś jeden wiersz uszkodzony (patrz `0 braków`)

Kolumny `Went Thru Tear Down?` i `Reject Desc` mogą być szczególnie ciekawe i nie należy ich odrzucać. Ewentualnie dla tych kolumn, w miejsce braków należy ustawić jakąś neutralną wartość.

In [27]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Timestamp,13506,2024-11-21 00:37:13.377362432,2024-10-01 06:09:27.163000,2024-10-26 03:08:16.290249984,2024-11-21 06:16:36.760000,2024-12-11 22:50:59.922749952,2025-01-17 05:21:41.107000,NaN
Pallet to Pallet CycleTime (Sec),13505.0,690.731877,23.0,159.0,273.0,500.0,1133689.0,11237.770507
Image,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Cycles,13506.0,1.017992,1.0,1.0,1.0,1.0,4.0,0.144664
Reject Code,13506.0,3.654894,0.0,0.0,0.0,0.0,273.0,29.679503
Nest ID Verify Cycles,13505.0,0.99926,0.0,1.0,1.0,1.0,2.0,0.029801
Nest ID Verify,13506.0,2.437731,0.0,1.0,2.0,3.0,4.0,1.147608
MARPOSS Judge Result,13505.0,1.142318,0.0,1.0,1.0,1.0,2.0,0.35108
M9 Stretch Dist RG Side,13505.0,168.012128,167.763,167.97,168.02,168.055,168.189,0.05956
"M81 DISTANCE 81,0 G",13505.0,81.001715,80.941,80.988,81.002,81.015,81.185,0.019062


# Podsumowanie

- `Serial Number` i `Part Number` zanieczyszczone nienumerycznymi danymi
- `Image` nienadaje się przez brak danych (NA w całej kolumnie)
- `Pallet to Pallet CycleTime (Sec)` - braki



---



# Szczegółowa analiza

Nie ma po co zaglądać tutaj :)

In [48]:
print(df['Status'].unique())
print(df['Status'].value_counts())

<StringArray>
['Accept', 'Reject']
Length: 2, dtype: string
Status
Accept    13304
Reject      202
Name: count, dtype: Int64


In [47]:
print(df['MARPOSS Judge Result'].unique())
print(df['MARPOSS Judge Result'].value_counts())

<IntegerArray>
[1, 2, 0, <NA>]
Length: 4, dtype: Int64
MARPOSS Judge Result
1    11567
2     1930
0        8
Name: count, dtype: Int64
